In [6]:
from user_api import *

In [7]:
import pandas as pd

try:
    df = pd.read_parquet("../data/raw/games.parquet")
    print("Cargado desde Parquet.")
except Exception:
    df = pd.read_pickle("data/raw/games.pkl")
    print("Cargado desde Pickle.")

print(df.shape)
df.head()

Cargado desde Parquet.
(122611, 43)


,app_id,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,...,positive,negative,estimated_owners,average_playtime_forever,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,tags
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0,0.00,0,,,,,...,0,0,0 - 0,0,0,0,0,0,0,[]
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0,5.24,0,"Springtime, April: when the cherry trees come ...","Springtime, April: when the cherry trees come ...","Spring has come, and our protagonist, Yukinari...",,...,252,3,0 - 20000,8,0,8,0,65,0,"{'Adventure': 27, 'Visual Novel': 19, 'Anime':..."
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0,4.99,0,"Immerse yourself in the most beloved, mystical...","Immerse yourself in the most beloved, mystical...",Discover an entrancing and spectacular world!,,...,21,3,0 - 20000,0,0,0,0,0,0,"{'Casual': 83, 'Card Game': 52, 'Solitaire': 4..."
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0,8.99,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...","synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",Yuha! I'll start the broadcast! Hakko's extrem...,,...,0,0,0 - 20000,0,0,0,0,0,1,[]
4,3631080,Maze Quest VR,"Apr 24, 2025",0,4.99,0,Its not just a Maze; its a Quest! Enter the ca...,Its not just a Maze; its a Quest! Enter the ca...,Its not just a Maze; its a Quest! Enter the ca...,,...,0,0,0 - 20000,0,0,0,0,0,0,[]


In [8]:
COLUMNS = [
    "tags",
    "genres",
    "categories"
]

In [9]:
import numpy as np

def es_vacio(x):

    if x is None:
        return True

    if isinstance(x, float) and np.isnan(x):
        return True

    if isinstance(x, (list, tuple)):
        return len(x) == 0

    if isinstance(x, np.ndarray):
        return x.size == 0

    if isinstance(x, str):
        return x.strip() == "" or x.strip() == "[]"

    return False

In [10]:
# Máscara de filas completamente vacías
mask_todas_vacias = df.apply(
    lambda row: all(es_vacio(row[col]) for col in COLUMNS),
    axis=1
)

# Filtrar esos juegos
juegos_sin_metadata = df[mask_todas_vacias]

print("📊 Juegos sin metadata en genres, tags y categories:")
print("Total:", len(juegos_sin_metadata))

# Mostrar solo nombres (primeros 50)
print("\n🎮 Primeros juegos sin metadata:")
juegos_sin_metadata[["app_id", "name", "tags", "genres", "categories"]].head(50)

📊 Juegos sin metadata en genres, tags y categories:
Total: 7984

🎮 Primeros juegos sin metadata:


,app_id,name,tags,genres,categories
0,2539430,Black Dragon Mage Playtest,[],[],[]
10,1946890,Codename: Warlock Playtest,[],[],[]
24,2349750,CyberVault Playtest,[],[],[]
25,2628280,A Night In Omar's Burger Playtest,[],[],[]
51,2689730,Dark and Deep Playtest,[],[],[]
97,2782210,Bill's Misfortune Playtest,[],[],[]
102,3664570,QUADRA CLASH Playtest,[],[],[]
116,3581240,Descendance Playtest,[],[],[]
118,1549730,Red Solstice 2: Survivors Playtest,[],[],[]
137,1639330,Pain Party Playtest,[],[],[]


# ESTO METER EN DATA CLEAN

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

COLUMNS = ["genres", "tags", "categories"]


import numpy as np
import ast

def limpiar_valor(x):

    if x is None:
        return ""

    # NaN
    if isinstance(x, float) and np.isnan(x):
        return ""

    # ndarray
    if isinstance(x, np.ndarray):
        return " ".join(str(i).strip() for i in x if str(i).strip())

    # lista real
    if isinstance(x, list):
        return " ".join(str(i).strip() for i in x if str(i).strip())

    # dict real
    if isinstance(x, dict):
        return " ".join(str(k).strip() for k in x.keys() if str(k).strip())

    # string
    if isinstance(x, str):

        texto = x.strip()

        if texto == "" or texto == "[]":
            return ""

        # 🔥 Si parece lista o dict, intentamos parsear
        if texto.startswith("[") or texto.startswith("{"):
            try:
                valor = ast.literal_eval(texto)

                if isinstance(valor, dict):
                    return " ".join(str(k).strip() for k in valor.keys() if str(k).strip())

                if isinstance(valor, list):
                    return " ".join(str(i).strip() for i in valor if str(i).strip())

            except:
                pass  # si falla, seguimos

        return texto

    return ""


def quitar_duplicados(texto):
    palabras = texto.split()
    return " ".join(dict.fromkeys(palabras))  # mantiene orden y elimina duplicados

def preparar_texto(df):

    df = df.copy()

    columnas_limpias = []

    for col in COLUMNS:
        if col in df.columns:

            serie = (
                df[col]
                .map(limpiar_valor)   # más rápido que apply
                .fillna("")
                .astype(str)
                .str.strip()
            )

            columnas_limpias.append(serie)

    if not columnas_limpias:
        df["combined_features"] = ""
        return df

    # concatenación vectorizada
    combined = columnas_limpias[0]

    for serie in columnas_limpias[1:]:
        combined = combined.str.cat(serie, sep=" ")

    # limpiar espacios extra
    df["combined_features"] = (
        combined
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .map(quitar_duplicados)
    )

    return df

df = preparar_texto(df)

print("📊 Filas con combined_features vacío:",
      (df["combined_features"] == "").sum())

# Opcional: ver cuáles son
print(df[df["combined_features"] == ""][["app_id","name"]].head(20))

vectorizer = TfidfVectorizer(stop_words="english")
X = vectorizer.fit_transform(df["combined_features"])

📊 Filas con combined_features vacío: 7984
      app_id                                         name
0    2539430                   Black Dragon Mage Playtest
10   1946890                   Codename: Warlock Playtest
24   2349750                          CyberVault Playtest
25   2628280            A Night In Omar's Burger Playtest
51   2689730                       Dark and Deep Playtest
97   2782210                   Bill's Misfortune Playtest
102  3664570                        QUADRA CLASH Playtest
116  3581240                         Descendance Playtest
118  1549730           Red Solstice 2: Survivors Playtest
137  1639330                          Pain Party Playtest
166  3392860                    Mechanical Pulse Playtest
186  3085750                                1971 Playtest
190  3973110                     Trickster Trove Playtest
236  1855530  From Warrior to Hero (Idle 3D RPG) Playtest
317  2205440                      Project OutFox Playtest
320  1761940                  

# PRUEBAS COMBINAR DATOS

In [12]:
df.loc[3, ["tags", "genres", "categories"]]

tags                                                         []
genres                              [Casual, Indie, Simulation]
categories    [Single-player, Steam Achievements, Family Sha...
Name: 3, dtype: object

In [13]:
print(type(df.loc[3, "categories"]))
print(df.loc[3, "categories"])

<class 'numpy.ndarray'>
['Single-player' 'Steam Achievements' 'Family Sharing']


In [14]:
print(type(df.loc[3, "combined_features"]))
print(df.loc[3, "combined_features"])

<class 'str'>
Casual Indie Simulation Single-player Steam Achievements Family Sharing


In [15]:
df.combined_features

0                                                          
1         Adventure Visual Novel Anime Cute Single-playe...
2         Casual Card Game Solitaire Puzzle Hidden Objec...
3         Casual Indie Simulation Single-player Steam Ac...
4         Action Early Access Single-player VR Only Stea...
                                ...                        
122606    Action Adventure Massively Multiplayer RPG Sim...
122607    Casual Strategy Free To Play Multi-player PvP ...
122608    Action Adventure Casual Early Access Single-pl...
122609    Action Adventure Casual Indie Single-player Fu...
122610    Action Indie Free To Play Single-player Partia...
Name: combined_features, Length: 122611, dtype: str

In [16]:
df.loc[2, ["tags", "genres", "categories", "combined_features"]]

tags                 {'Casual': 83, 'Card Game': 52, 'Solitaire': 4...
genres                                                        [Casual]
categories                             [Single-player, Family Sharing]
combined_features    Casual Card Game Solitaire Puzzle Hidden Objec...
Name: 2, dtype: object

In [17]:
import re

def auditar_combined_features(df):

    print("🔎 INICIANDO AUDITORÍA\n")

    total = len(df)

    # 1️⃣ Tipo incorrecto
    no_string = df[~df["combined_features"].apply(lambda x: isinstance(x, str))]
    print("❌ Filas que NO son string:", len(no_string))

    # 2️⃣ Contienen sintaxis de dict/lista sin limpiar
    patron_estructura = r"[\{\}\[\]:]"
    estructura_rara = df[df["combined_features"].str.contains(patron_estructura, regex=True, na=False)]
    print("❌ Filas con `{ }`, `[ ]` o `:` :", len(estructura_rara))

    # 3️⃣ Dobles espacios
    dobles_espacios = df[df["combined_features"].str.contains(r"\s{2,}", regex=True, na=False)]
    print("⚠️ Filas con dobles espacios:", len(dobles_espacios))

    # 4️⃣ Texto literal problemático
    texto_raro = df[
        df["combined_features"].str.contains(r"\b(None|nan)\b", regex=True, na=False)
    ]
    print("❌ Filas con 'None' o 'nan' literal:", len(texto_raro))

    # 5️⃣ Vacías
    vacias = df[df["combined_features"].str.strip() == ""]
    print("ℹ️ Filas vacías:", len(vacias))

    print("\n📊 RESUMEN:")
    print("Total filas:", total)
    print("Porcentaje vacías:", round(len(vacias)/total*100, 2), "%")

    print("\n🔎 Ejemplos problemáticos (si existen):")

    if len(estructura_rara) > 0:
        print("\nEjemplo estructura rara:")
        print(estructura_rara["combined_features"].head(3))

    if len(dobles_espacios) > 0:
        print("\nEjemplo dobles espacios:")
        print(dobles_espacios["combined_features"].head(3))

    if len(texto_raro) > 0:
        print("\nEjemplo texto raro:")
        print(texto_raro["combined_features"].head(3))

    print("\n✅ Auditoría finalizada.")

In [18]:
auditar_combined_features(df)

🔎 INICIANDO AUDITORÍA

❌ Filas que NO son string: 0
❌ Filas con `{ }`, `[ ]` o `:` : 0
⚠️ Filas con dobles espacios: 0
❌ Filas con 'None' o 'nan' literal: 0
ℹ️ Filas vacías: 7984

📊 RESUMEN:
Total filas: 122611
Porcentaje vacías: 6.51 %

🔎 Ejemplos problemáticos (si existen):

✅ Auditoría finalizada.


C:\Users\guarm\AppData\Local\Temp\ipykernel_42732\1306733024.py:24: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["combined_features"].str.contains(r"\b(None|nan)\b", regex=True, na=False)


In [19]:
import numpy as np

def construir_vector_usuario(df, juegos_usuario, X):

    if X is None:
        return None

    df["app_id"] = pd.to_numeric(df["app_id"], errors="coerce")

    appids_usuario = juegos_usuario["appid"].astype(int).tolist()
    indices = df[df["app_id"].isin(appids_usuario)].index

    if len(indices) == 0:
        return None

    # promedio
    user_vector = X[indices].mean(axis=0)

    # 🔥 CONVERTIMOS A ARRAY NORMAL
    user_vector = np.asarray(user_vector)

    return user_vector

al recomendar un juego asegurarme que no recomiende uno que el usuario ya tenga

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

def recomendar(df, X, user_vector, juegos_usuario, top_n=10):

    if X is None or user_vector is None:
        print("❌ X o user_vector son None")
        return pd.DataFrame()

    # asegurar tipos numéricos
    df["app_id"] = pd.to_numeric(df["app_id"], errors="coerce")

    # calcular similaridad
    similarities = cosine_similarity(user_vector, X)
    scores = similarities.flatten()

    df = df.copy()
    df["similarity"] = scores

    # 🔹 obtener juegos del usuario
    appids_usuario = set(juegos_usuario["appid"].astype(int))

    # 🔹 no recomendar juegos que el usuario ya posee
    recomendaciones = df[~df["app_id"].isin(appids_usuario)]

    # ordenar por similaridad
    recomendaciones = recomendaciones.sort_values(
        "similarity", ascending=False
    )

    return recomendaciones.head(top_n)

In [21]:
print(df["combined_features"].head())

0                                                     
1    Adventure Visual Novel Anime Cute Single-playe...
2    Casual Card Game Solitaire Puzzle Hidden Objec...
3    Casual Indie Simulation Single-player Steam Ac...
4    Action Early Access Single-player VR Only Stea...
Name: combined_features, dtype: str


In [22]:
df["app_id"]

0         2539430
1          496350
2         1034400
3         3292190
4         3631080
           ...   
122606    4152910
122607    4042800
122608    3522550
122609    3680350
122610    4141790
Name: app_id, Length: 122611, dtype: str

In [23]:
juegos_usuario = obtener_juegos_usuario(76561198215235853)
juegos_usuario

2026-03-18 09:44:30,789 - INFO - Consultando juegos del usuario 76561198215235853
2026-03-18 09:44:30,789 - INFO - Enviando petición a la API de Steam
2026-03-18 09:44:31,110 - INFO - Código de respuesta: 200
2026-03-18 09:44:31,110 - INFO - Se encontraron 63 juegos


,appid,name,playtime_forever,img_icon_url,has_community_visible_stats,content_descriptorids,playtime_2weeks,has_leaderboards
0,550,Left 4 Dead 2,283,7d5a243f9500d2f8467312822f8af2a2928777ed,True,"[2, 5]",NaN,NaN
1,730,Counter-Strike 2,12877,8dbc71957312bbd3baea65848b545be9eae2a355,True,"[2, 5]",NaN,NaN
2,1930,Two Worlds: Epic Edition,0,f20291118f8abb2c4d50579801b908720b7d2651,NaN,NaN,NaN,NaN
3,4000,Garry's Mod,111,4a6f25cfa2426445d0d9d6e233408de4d371ce8b,True,NaN,NaN,NaN
4,7670,BioShock,0,9a7c9f640a76e6a32592277dbbc36a0f6da05372,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
58,2246340,Monster Hunter Wilds,24654,23474d61e4506351f9b26f230fd417d017cf2806,True,NaN,906.0,NaN
59,2429640,THRONE AND LIBERTY,32,f03eb7d1aee8d7997b28ccc8de047b8fe561a657,True,"[2, 5]",NaN,NaN
60,2767030,Marvel Rivals,226,839b4712925b95702ca56e0c4d399adf54f4d617,True,NaN,NaN,NaN
61,2796010,Party Club,0,a80c96dde162ef5ed25704384b6e312b6b6cddf9,True,NaN,NaN,NaN


In [24]:
print("Tipo appID en dataframe:", df["app_id"].dtype)

print("Tipo primer appid usuario:", type(juegos_usuario.iloc[0]["appid"]))

Tipo appID en dataframe: str
Tipo primer appid usuario: <class 'numpy.int64'>


In [25]:
construir_perfil_usuario(df, juegos_usuario)

2026-03-18 09:44:31,139 - INFO - Juegos con al menos 120 minutos jugados: 29
2026-03-18 09:44:31,139 - INFO - AppIDs válidos: 29
2026-03-18 09:44:31,144 - INFO - Juegos encontrados en dataset: 28


,app_id,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,...,negative,estimated_owners,average_playtime_forever,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,tags,combined_features
1423,2246340,Monster Hunter Wilds,"Feb 27, 2025",0,38.49,190,Deluxe Edition and Premium Deluxe Edition ■Mon...,The unbridled force of nature runs wild and re...,The unbridled force of nature runs wild and re...,,...,91142,20000000 - 50000000,4590,515,3250,281,45,12494,"{'Hunting': 2273, 'Action': 1946, 'Multiplayer...",Action Adventure RPG Hunting Multiplayer Onlin...
4117,1097150,Fall Guys,"Aug 3, 2020",0,0.00,0,You’re invited to dive and dodge your way to v...,You’re invited to dive and dodge your way to v...,"Fall Guys is a free, cross-platform massively ...",,...,88589,20000000 - 50000000,1512,81,553,55,0,664,"{'Multiplayer': 1604, 'Party Game': 1539, 'Bat...",Action Casual Indie Massively Multiplayer Spor...
8773,513710,SCUM,"Jun 17, 2025",0,17.99,13,1.0 Launch Road to 1.0 and beyond! Check out w...,Welcome prisoners to SCUM Island! Where partic...,"Traverse punishing environments, looting, craf...","“Scum has a rare crafting system, a wonder of ...",...,28411,5000000 - 10000000,5771,1375,888,1534,60,8535,"{'Survival': 1480, 'Open World Survival Craft'...",Action Adventure Indie Massively Multiplayer S...
8878,578080,PUBG: BATTLEGROUNDS,"Dec 21, 2017",13,0.00,0,LAND Drop into an ever-growing and changin...,LAND Drop into an ever-growing and changin...,"PUBG: BATTLEGROUNDS, the high-stakes winner-ta...",,...,1037487,100000000 - 200000000,23787,730,6082,302,0,314682,"{'Survival': 14893, 'Shooter': 12788, 'Battle ...",Action Adventure Massively Multiplayer Free To...
14153,346110,ARK: Survival Evolved,"Aug 27, 2017",0,9.89,18,ARK: Survival Ascended! Respawn into a new din...,"As a man or woman stranded naked, freezing and...","Stranded on the shores of a mysterious island,...",,...,117993,20000000 - 50000000,11685,940,921,627,34,22170,"{'Open World Survival Craft': 12961, 'Survival...",Action Adventure Indie Massively Multiplayer R...
17066,252950,Rocket League®,"Jul 6, 2015",0,0.00,0,Rocket League is a high-powered hybrid of arca...,Rocket League is a high-powered hybrid of arca...,Rocket League is a high-powered hybrid of arca...,,...,73775,10000000 - 20000000,19487,491,2807,249,0,21106,"{'Multiplayer': 6610, 'Football (Soccer)': 524...",Action Indie Racing Sports Multiplayer Footbal...
17645,389730,TEKKEN 7,"Jun 1, 2017",0,9.99,24,GET READY Definitive Edition Includes: - TEKKE...,Discover the epic conclusion of the Mishima cl...,Discover the epic conclusion of the long-time ...,,...,15580,2000000 - 5000000,3473,81,551,41,75,599,"{'Fighting': 1279, 'Action': 675, 'Multiplayer...",Action Sports Fighting Multiplayer Competitive...
21033,1599340,Lost Ark,"Feb 11, 2022",17,0.00,7,Embark on an odyssey for the Lost Ark in a vas...,Embark on an odyssey for the Lost Ark in a vas...,Embark on an odyssey for the Lost Ark in a vas...,,...,58540,50000000 - 100000000,5246,1846,746,335,0,17355,"{'MMORPG': 438, 'Free to Play': 380, 'Action R...",Action Adventure Massively Multiplayer RPG Fre...
24982,502500,ACE COMBAT™ 7: SKIES UNKNOWN,"Jan 31, 2019",0,4.79,21,ACE COMBAT™ 7: SKIES UNKNOWN - TOP GUN: Maveri...,Purchase ACE COMBAT™ 7: SKIES UNKNOWN and get ...,Become an ace pilot and soar through photoreal...,,...,5990,2000000 - 5000000,1336,200,429,156,92,340,"{'Flight': 845, 'Jet': 643, 'Military': 610, '...",Action Simulation Flight Jet Military War Shoo...
35608,275850,No Man's Sky,"Aug 12, 2016",0,23.99,1,Inspired by the adventure and imagination that...,Inspired by the adventure and imagination that...,No Man's Sky is a game about exploration and s...,“utterly breathtaking” 9/10 – GameSpot “Soulfu...,...,59526,5000000 - 10000000,3401,810,1239,144,0,12291,"{'Open World': 5393, 'Open World Survival Craf...",Action Adventure Open World Survival Craft Spa...


In [26]:
# =====================================================
# EJEMPLO DE USO
# =====================================================

steam_url = "https://steamcommunity.com/profiles/76561198367022349/"

if __name__ == "__main__":

    import pandas as pd
    
    # 1️⃣ Cargar dataset
    print("🔄 Cargando dataset...")
    
    # 2️⃣ Preparar texto y matriz TF-IDF
    print("🧠 Preparando features...")
    df = preparar_texto(df)

    from sklearn.feature_extraction.text import TfidfVectorizer
    vectorizer = TfidfVectorizer(stop_words="english")
    X = vectorizer.fit_transform(df["combined_features"])

    # 3️⃣ Pedir URL de Steam
    url_usuario = input("\nIntroduce tu perfil de Steam: ")

    steamid = extraer_steamid(url_usuario)

    if not steamid:
        print("❌ No se pudo obtener el SteamID.")
        exit()

    print("✅ SteamID detectado:", steamid)

    # 4️⃣ Obtener juegos del usuario
    juegos_usuario = obtener_juegos_usuario(steamid)

    if juegos_usuario.empty:
        print("⚠️ Perfil privado o sin juegos.")
        exit()

    print(f"🎮 Juegos encontrados: {len(juegos_usuario)}")

    # 5️⃣ Construir vector del usuario
    user_vector = construir_vector_usuario(df, juegos_usuario, X)

    if user_vector is None:
        print("⚠️ No se pudieron mapear los juegos al dataset.")
        exit()
    else: 
        print("Conseguido")

    # 6️⃣ Generar recomendaciones
    recomendaciones = recomendar(df, X, user_vector, juegos_usuario, top_n=10)

    print("\n🎯 RECOMENDACIONES PERSONALIZADAS:\n")

    for i, row in recomendaciones.iterrows():
        print(f"🎮 {row['name']}")
        print(f"   Similaridad: {row['similarity']:.4f}")
        print(f"   Géneros: {row['genres']}")
        print(f"   Tags: {row['tags']}")
        print(f"   Categorias: {row['categories']}")


        print("-" * 50)

🔄 Cargando dataset...
🧠 Preparando features...


2026-03-18 09:44:54,908 - INFO - SteamID numérico detectado
2026-03-18 09:44:54,908 - INFO - Consultando juegos del usuario 76561198367022349
2026-03-18 09:44:54,909 - INFO - Enviando petición a la API de Steam


✅ SteamID detectado: 76561198367022349


2026-03-18 09:44:55,302 - INFO - Código de respuesta: 200
2026-03-18 09:44:55,302 - INFO - Se encontraron 76 juegos


🎮 Juegos encontrados: 76
Conseguido

🎯 RECOMENDACIONES PERSONALIZADAS:

🎮 DARK SOULS™ III
   Similaridad: 0.6445
   Géneros: ['Action']
   Tags: {'Souls-like': 9604, 'Dark Fantasy': 7905, 'Difficult': 6967, 'RPG': 5966, 'Atmospheric': 5311, 'Lore-Rich': 4544, 'Third Person': 4272, 'Exploration': 3797, 'Story Rich': 3779, 'Action RPG': 3576, 'Co-op': 3499, 'Great Soundtrack': 3418, 'Adventure': 3307, 'Action': 3292, 'Multiplayer': 3214, 'PvP': 3161, 'Open World': 3080, 'Singleplayer': 2427, 'Character Customization': 1936, 'Replay Value': 1926}
   Categorias: ['Single-player' 'Multi-player' 'Co-op' 'Steam Achievements'
 'Full controller support' 'Steam Trading Cards' 'Camera Comfort'
 'Custom Volume Controls' 'Playable without Timed Input' 'Save Anytime'
 'Stereo Sound' 'Surround Sound' 'Remote Play on Phone'
 'Remote Play on Tablet' 'Remote Play on TV' 'Family Sharing']
--------------------------------------------------
🎮 FINAL FANTASY XV WINDOWS EDITION
   Similaridad: 0.6403
   Géner

In [44]:
url_usuario = "https://steamcommunity.com/profiles/76561198367022349"

print("🔄 Preparando texto...")
df = preparar_texto(df)
print("✅ combined_features creado")

print("🧠 Creando TF-IDF...")
vectorizer = TfidfVectorizer(stop_words="english")
X = vectorizer.fit_transform(df["combined_features"])
print(f"✅ TF-IDF listo. Shape: {X.shape}")

print("\n🔗 Obteniendo SteamID...")
url_usuario = input("Introduce tu URL de Steam: ")
steamid = extraer_steamid(url_usuario)

if not steamid:
    print("❌ No se pudo obtener SteamID")
    exit()

print(f"✅ SteamID: {steamid}")

print("🎮 Obteniendo juegos del usuario...")
juegos_usuario = obtener_juegos_usuario(steamid)

if not juegos_usuario:
    print("⚠️ No se encontraron juegos o perfil privado")
    exit()

print(f"✅ Juegos encontrados: {len(juegos_usuario)}")

print("🧠 Construyendo vector del usuario...")
user_vector = construir_vector_usuario(df, juegos_usuario, X)

if user_vector is None:
    print("❌ No se pudo construir vector del usuario")
    exit()

print("✅ Vector del usuario creado")

print("🎯 Generando recomendaciones...")
recomendaciones = recomendar(df, X, user_vector, juegos_usuario)

if recomendaciones is None or recomendaciones.empty:
    print("⚠️ No se encontraron recomendaciones")
else:
    print("\n🎉 RECOMENDACIONES:")
    print(recomendaciones[["name", "similarity"]].head(10))

🔄 Preparando texto...
✅ combined_features creado
🧠 Creando TF-IDF...
✅ TF-IDF listo. Shape: (122611, 544)

🔗 Obteniendo SteamID...


2026-03-15 22:07:42,331 - INFO - SteamID numérico detectado
2026-03-15 22:07:42,333 - INFO - Consultando juegos del usuario 76561198367022349
2026-03-15 22:07:42,333 - INFO - Enviando petición a la API de Steam


✅ SteamID: 76561198367022349
🎮 Obteniendo juegos del usuario...


2026-03-15 22:07:42,630 - INFO - Código de respuesta: 200
2026-03-15 22:07:42,631 - INFO - Se encontraron 76 juegos


ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [ ]:
import numpy as np

def construir_vector_usuario(df, juegos_usuario, feature_matrix):
    
    appids_usuario = [j["appid"] for j in juegos_usuario]
    
    indices = df[df["appID"].isin(appids_usuario)].index
    
    if len(indices) == 0:
        return None
    
    # Vector promedio
    user_vector = np.asarray(feature_matrix[indices].mean(axis=0))
    
    return user_vector

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df.genres)